In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit provide or receive gifts or anything of 
   value to/from OR pay for the travel, hospitality or entertainment of Customers 
   and Third Parties?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. TARGET CATEGORY CODES: Hardcoded explicitly to (066, 009, 012, 067, 095, 134, 
      140, 180, 181, 793). Cast to INT to avoid string leading-zero mismatch issues.
   3. COUPA DATA (7 Files): Unions the 7 Coupa files. Parses the 'Account' string 
      (format: xxxx-xxxx-xxxxxx) to extract the Cost Center and Category Code. 
   4. FINANCE DATA (4 Files verified): Unions the Finance files. Extracts the Cost 
      Center and Category Code directly.
   5. CONSOLIDATION: Combines Coupa and Finance data into one master list. Filters 
      strictly for the Target Category Codes.
   6. CC MAPPING & THE 793 RULE: Maps to AU_IDs. Applies the exception rule that 
      Category Code 793 is ONLY valid for AU 101016.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    -- 2a. Union all 7 Coupa files into one master dataset
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 2b. Parse the Coupa Account string: splits by '-' 
    SELECT 
        -- Cast to INT strips leading zeros naturally to help matching
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- 3a. Union all available Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 3b. Clean the Finance columns
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 4a. Stack Coupa and Finance data together
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Finance_Parsed
),

Flagged_Transactions AS (
    -- 4b. Strictly keep transactions matching the exact Target Category Codes
    SELECT 
        Cleaned_CC,
        CatCode_Int,
        Source_System
    FROM All_Transactions
    WHERE Cleaned_CC IS NOT NULL
      AND CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793)
),

CC_Mapping AS (
    -- 5a. Grab the CC Bootstrap Mapping
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 5b. Map transactions to AU_IDs and APPLY THE 793 RULE
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- THE 793 EXCEPTION RULE: Exclude code 793 if the AU is NOT 101016
        NOT (f.CatCode_Int = 793 AND m.AU_ID != '101016')
    GROUP BY m.AU_ID
)

-- 6. Final Output
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA01' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 - Travel, Lodging, and Entertainment Traceability
   
   PURPOSE: 
   Isolates the data flow for EBA01 to show exactly how Coupa and Finance records 
   are parsed, whether their Category Codes trigger a flag, how they map to an AU, 
   and whether they get blocked by the strict "793" exception rule.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Combined_Coupa_Raw AS (
    -- Union all 7 Coupa files
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- Parse the Coupa string into CC and Category Code
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- Union all 6 Finance files
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- Standardize Finance columns
    SELECT 
        CostCenter AS Raw_String,
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- Stack them together
    SELECT Raw_String, Cleaned_CC, CatCode_Int, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode_Int, Source_System FROM Finance_Parsed
),

CC_Mapping AS (
    -- Grab Bootstrap mappings
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    t.Source_System,
    t.Raw_String AS Original_Record,
    t.Cleaned_CC AS Parsed_Cost_Center,
    t.CatCode_Int AS Parsed_Category_Code,
    
    -- Flag if it hits the Master target list
    CASE WHEN t.CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793) 
         THEN 'YES' ELSE 'NO' END AS Is_Target_Category,
         
    m.AU_ID AS Mapped_AU_ID,
    mast.Master_AU_Name,
    
    -- Trace exactly how the 793 rule and mapping is applied
    CASE 
        WHEN t.CatCode_Int = 793 AND m.AU_ID != '101016' THEN '❌ BLOCKED: 793 only valid for 101016'
        WHEN t.CatCode_Int = 793 AND m.AU_ID = '101016' THEN '✅ KEPT: Valid 793 mapping'
        WHEN t.CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181) THEN '✅ KEPT: Standard Category'
        ELSE '❌ DROPPED: Not a target category'
    END AS Pipeline_Status,
    
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Master_List_Status
    
FROM All_Transactions t
LEFT JOIN CC_Mapping m 
    ON t.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.Master_Numeric_ID

-- Optional: Uncomment the line below to only view the flagged transactions
-- WHERE t.CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793)
ORDER BY t.Cleaned_CC;